# Optuna Tuning Basics

This notebook introduces how to use **Optuna** as a tuning backend in VAMOS via `ModelBasedTuner`.

We will:
1. Define a parameter space for NSGA-II
2. Create a tuning task with instances and seeds
3. Run Optuna TPE to find good hyperparameters
4. Inspect and visualize results

### Key concepts

| Concept | Description |
|---|---|
| `n_trials` (max_trials) | Number of hyperparameter configurations Optuna will try |
| `budget_per_run` | Function evaluations the MOEA runs for **each** configuration |
| `eval_fn` | Your function that runs the MOEA and returns a quality score |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from vamos import optimize
from vamos.foundation.quality_indicators import compute_hypervolume
from vamos.engine.tuning import (
    ModelBasedTuner,
    TuningTask,
    Instance,
    EvalContext,
    ParamSpace,
    Real,
    Int,
    Categorical,
    config_from_assignment,
)

plt.style.use("ggplot")
print("Imports OK")

## 1. Define the Parameter Space

We specify which hyperparameters Optuna can explore and their ranges.

In [ ]:
param_space = ParamSpace(
    params={
        "pop_size": Int("pop_size", 50, 200),
        "crossover_prob": Real("crossover_prob", 0.6, 1.0),
        "crossover_eta": Real("crossover_eta", 5.0, 30.0),
        "mutation_eta": Real("mutation_eta", 5.0, 30.0),
    }
)

# Preview random samples
rng = np.random.default_rng(42)
for i in range(3):
    print(f"Sample {i+1}: {param_space.sample(rng)}")

## 2. Define the Evaluation Function

This function is called once per trial. It receives a configuration dict and an `EvalContext`, runs NSGA-II, and returns a scalar score (hypervolume).

In [ ]:
REF_POINT = np.array([1.1, 1.1])


def eval_fn(config: dict, ctx: EvalContext) -> float:
    """Run NSGA-II with the given config and return hypervolume."""
    cfg = config_from_assignment("nsgaii", config)

    result = optimize(
        ctx.instance.name,
        algorithm="nsgaii",
        algorithm_config=cfg,
        max_evaluations=ctx.budget,
        seed=ctx.seed,
        n_var=ctx.instance.n_var,
        engine="numpy",
    )

    if result.F is None or len(result.F) == 0:
        return 0.0
    return compute_hypervolume(result.F, REF_POINT)


print("eval_fn defined")

## 3. Create the Tuning Task

A `TuningTask` bundles together:
- **param_space**: what hyperparameters to search
- **instances**: which problem(s) to evaluate on
- **seeds**: how many independent runs per configuration
- **budget_per_run**: function evaluations the MOEA gets per run
- **aggregator**: how to combine scores across instances and seeds

In [ ]:
task = TuningTask(
    name="basic_optuna_demo",
    param_space=param_space,
    instances=[Instance(name="zdt1", n_var=10, kwargs={})],
    seeds=[0, 1, 2],          # 3 seeds per config
    budget_per_run=3000,      # 3000 FEs per run (small for demo)
    maximize=True,            # maximize hypervolume
    aggregator=lambda scores: float(np.mean(scores)),
)

print(f"Task: {task.name}")
print(f"  Instances: {[i.name for i in task.instances]}")
print(f"  Seeds: {list(task.seeds)}")
print(f"  Budget per run: {task.budget_per_run} FEs")

## 4. Run Optuna with `ModelBasedTuner`

We create a `ModelBasedTuner` with:
- `max_trials=20`: try 20 different configurations
- `backend="optuna"`: use Optuna
- `optuna_sampler="tpe"`: Tree-structured Parzen Estimator

Each trial evaluates a config across 1 instance × 3 seeds = **3 MOEA runs**.
Total MOEA runs: 20 trials × 3 = 60 runs.

In [ ]:
tuner = ModelBasedTuner(
    task=task,
    max_trials=20,          # 20 configurations to try
    backend="optuna",
    optuna_sampler="tpe",   # TPE sampler
    seed=42,
    n_jobs=1,               # sequential (set >1 for parallel)
)

print("Starting Optuna tuning...")
best_config, history = tuner.run(eval_fn)

print(f"\nDone! Evaluated {len(history)} configurations.")
print(f"\nBest configuration:")
for k, v in best_config.items():
    print(f"  {k}: {v}")

## 5. Inspect Results

The `history` is a list of `TrialResult` objects with `.trial_id`, `.config`, `.score`, and `.details`.

In [ ]:
# Best score
scores = [t.score for t in history]
best_score = max(scores)
print(f"Best HV: {best_score:.6f}")
print(f"Worst HV: {min(scores):.6f}")
print(f"Mean HV: {np.mean(scores):.6f}")

# Convergence plot: best-so-far across trials
best_so_far = np.maximum.accumulate(scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(scores, "o-", alpha=0.6, label="Trial score")
axes[0].plot(best_so_far, "r-", lw=2, label="Best so far")
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("Hypervolume")
axes[0].set_title("Optuna Tuning Progress")
axes[0].legend()

# Score distribution
axes[1].hist(scores, bins=10, edgecolor="black", alpha=0.7)
axes[1].axvline(best_score, color="red", ls="--", label=f"Best: {best_score:.4f}")
axes[1].set_xlabel("Hypervolume")
axes[1].set_ylabel("Count")
axes[1].set_title("Score Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Validate the Best Configuration

Run the best config on a fresh seed with a larger budget to confirm it's actually good.

In [ ]:
# Validation with a fresh seed and larger budget
val_ctx = EvalContext(
    instance=Instance(name="zdt1", n_var=10, kwargs={}),
    seed=999,
    budget=10000,
)

hv_tuned = eval_fn(best_config, val_ctx)

# Compare with a default config
default_config = {
    "pop_size": 100,
    "crossover_prob": 0.9,
    "crossover_eta": 20.0,
    "mutation_eta": 20.0,
}
hv_default = eval_fn(default_config, val_ctx)

print(f"Tuned config HV:   {hv_tuned:.6f}")
print(f"Default config HV: {hv_default:.6f}")
print(f"Improvement:       {hv_tuned - hv_default:+.6f}")

## Summary

| Step | What we did |
|---|---|
| 1 | Defined a `ParamSpace` with `Int`, `Real` parameters |
| 2 | Wrote an `eval_fn` that runs NSGA-II and returns hypervolume |
| 3 | Created a `TuningTask` with instances, seeds, and budget |
| 4 | Ran `ModelBasedTuner` with Optuna TPE for 20 trials |
| 5 | Inspected scores and plotted convergence |
| 6 | Validated the best config against a default |

**Next**: See `18_optuna_tuning_intermediate.ipynb` for multi-problem tuning and sampler comparison.